# AICA adapter training (QLoRA, BRD 13)

Trains a LoRA adapter for the approved base model `qwen2.5-coder-7b` (Ollama
`qwen2.5-coder:7b`, built from `Qwen/Qwen2.5-Coder-7B-Instruct`) on a dataset built by
`aica adapt build`, with the hyperparameters of the job spec written by `aica adapt plan`.

**Inputs** (one private Kaggle dataset, uploaded by `training/kaggle/upload.py`):
`train.jsonl`, `manifest.json` and `job.json`.

**Outputs** (in `/kaggle/working/out`): `adapter/` (PEFT safetensors), `adapter.gguf` (for
Ollama's `ADAPTER`), `provenance.json` and `training_log.json`.

Nothing here needs a secret: the base model is public and the dataset is attached to the
notebook. Settings: *Accelerator* GPU T4 x2 (or P100), *Internet* on.

In [ ]:
# Pinned so the run is reproducible; Kaggle's image already has torch.
!pip install -q "transformers==4.56.2" "peft==0.17.1" "bitsandbytes==0.50.2" "accelerate==1.10.1" "sentencepiece==0.2.1"

In [ ]:
import glob, hashlib, json, os, platform, subprocess, sys, time
from pathlib import Path

matches = glob.glob("/kaggle/input/**/job.json", recursive=True)
assert len(matches) == 1, f"attach exactly one AICA training dataset (found {matches})"
INPUT = Path(matches[0]).parent
OUT = Path("/kaggle/working/out")
OUT.mkdir(parents=True, exist_ok=True)

job = json.loads((INPUT / "job.json").read_text())
manifest = json.loads((INPUT / "manifest.json").read_text())
cfg = job["config"]

# The adapter is only valid for the exact weights it is trained on (MM-013). This notebook
# knows one base; anything else is refused rather than trained on the wrong model.
BASES = {"qwen2.5-coder:7b": "Qwen/Qwen2.5-Coder-7B-Instruct"}
base_version = job["base_model"]["version"]
assert base_version in BASES, f"no Hugging Face weights known for {base_version!r}"
BASE_REPO = BASES[base_version]
assert cfg["method"] == "qlora", "this notebook trains QLoRA adapters only"

# The dataset must be byte-for-byte the one the job was planned against.
body = (INPUT / "train.jsonl").read_bytes()
digest = hashlib.sha256(body).hexdigest()
assert digest == manifest["sha256"] == job["dataset"]["sha256"], "dataset does not verify"
examples = [json.loads(line) for line in body.decode("utf-8").splitlines() if line.strip()]
assert len(examples) == manifest["examples"]
print(f"job {job['job']}: {len(examples)} examples, base {BASE_REPO}, config {cfg}")

In [ ]:
import torch
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

torch.manual_seed(cfg["seed"])
# Pin the exact revision downloaded, so provenance names the weights, not a moving branch.
base_path = snapshot_download(BASE_REPO)
BASE_REVISION = Path(base_path).name
tokenizer = AutoTokenizer.from_pretrained(base_path)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()  # T4/P100: fp16
dtype = torch.bfloat16 if bf16 else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    base_path,
    device_map={"": 0},
    dtype=dtype,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=dtype,
    ),
)
print(BASE_REPO, BASE_REVISION, torch.cuda.get_device_name(0), dtype)

In [ ]:
# Loss on the assistant's answer only: the model learns to produce plans and arguments,
# not to reproduce the prompts or the repository content inside them.
MAX_LEN = cfg["max_seq_length"]

def encode(example):
    messages = example["messages"]
    assert messages[-1]["role"] == "assistant"
    prompt = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    answer = messages[-1]["content"] + "<|im_end|>\n"
    p = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    a = tokenizer(answer, add_special_tokens=False)["input_ids"]
    return {"input_ids": p + a, "labels": [-100] * len(p) + a}

encoded, too_long = [], 0
for ex in examples:
    item = encode(ex)
    if len(item["input_ids"]) > MAX_LEN:
        too_long += 1  # dropped, never truncated: a cut answer teaches a broken plan
        continue
    encoded.append(item)
lengths = sorted(len(e["input_ids"]) for e in encoded)
print(f"{len(encoded)} examples kept, {too_long} longer than {MAX_LEN} tokens dropped; "
      f"median {lengths[len(lengths) // 2]} tokens, max {lengths[-1]}")
assert encoded, "no example fits max_seq_length"


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(
    model,
    LoraConfig(
        r=cfg["lora_rank"],
        lora_alpha=cfg["lora_alpha"],
        lora_dropout=cfg["lora_dropout"],
        target_modules=cfg["target_modules"],
        task_type="CAUSAL_LM",
    ),
)
model.print_trainable_parameters()

def collate(batch):
    width = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id
    ids = [b["input_ids"] + [pad] * (width - len(b["input_ids"])) for b in batch]
    labels = [b["labels"] + [-100] * (width - len(b["labels"])) for b in batch]
    mask = [[1] * len(b["input_ids"]) + [0] * (width - len(b["input_ids"])) for b in batch]
    return {"input_ids": torch.tensor(ids), "labels": torch.tensor(labels),
            "attention_mask": torch.tensor(mask)}

args = TrainingArguments(
    output_dir="/kaggle/working/checkpoints",
    num_train_epochs=cfg["epochs"],
    learning_rate=cfg["learning_rate"],
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    lr_scheduler_type="cosine",
    warmup_steps=max(1, round(0.05 * cfg['epochs'] * -(-len(encoded) // 8))),
    logging_steps=1,
    save_strategy="no",
    fp16=not bf16,
    bf16=bf16,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    seed=cfg["seed"],
    report_to=[],
)
started = time.time()
trainer = Trainer(model=model, args=args, train_dataset=encoded, data_collator=collate)
result = trainer.train()
print(result, f"{(time.time() - started) / 60:.1f} min")

In [ ]:
# Save the adapter and a GGUF copy of it for Ollama (FROM qwen2.5-coder:7b + ADAPTER).
adapter_dir = OUT / "adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
(OUT / "training_log.json").write_text(json.dumps(trainer.state.log_history, indent=1))

gguf_error = None
try:
    if not Path("/kaggle/working/llama.cpp").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp",
                        "/kaggle/working/llama.cpp"], check=True)
    subprocess.run([sys.executable, "/kaggle/working/llama.cpp/convert_lora_to_gguf.py",
                    str(adapter_dir), "--base", base_path, "--outfile", str(OUT / "adapter.gguf"),
                    "--outtype", "f16"], check=True)
except Exception as exc:  # the safetensors adapter is still usable; convert locally instead
    gguf_error = repr(exc)
    print("GGUF conversion failed:", gguf_error)

In [ ]:
def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

files = sorted(p for p in OUT.rglob("*") if p.is_file() and p.name != "provenance.json")
provenance = {
    "job": job["job"],
    "config_version": job["config_version"],
    "base_model": job["base_model"],
    "base_weights": {"repo": BASE_REPO, "revision": BASE_REVISION},
    "dataset": {"version": manifest["version"], "sha256": digest, "examples": len(examples),
                "trained_on": len(encoded), "dropped_too_long": too_long},
    "train_loss": result.training_loss,
    "gguf_error": gguf_error,
    "files": {str(p.relative_to(OUT)): sha256_of(p) for p in files},
    "environment": {"python": platform.python_version(), "torch": torch.__version__,
                    "gpu": torch.cuda.get_device_name(0)},
}
(OUT / "provenance.json").write_text(json.dumps(provenance, indent=1))
print(json.dumps({k: v for k, v in provenance.items() if k != "files"}, indent=1))